# Import libraries

In [1]:
import torch
from torch.utils.data import Subset, Dataset, DataLoader
from tqdm import tqdm
import multiprocessing as mp

from models.gpt import GPT
from dataset.dataset import OthelloDataset
from game.othello import GameBoard, Piece
from models.linear_probe import LinearProbe

# Load  a target model

In [41]:
def load_checkpoint(model, checkpoint):
    checkpoint = torch.load("../checkpoints/" + checkpoint)
    model.load_state_dict(checkpoint)

model = GPT()
model_name = "gpt_batch_2400_loss_2.0949_pass_rate_96.3701"
load_checkpoint(model, model_name + ".pt")

# Load the dataset

In [42]:
NUM_OF_GAMES_USED_FOR_INTERVENTIONS = 10000
NUM_OF_GAMES_FOR_PROBE_TRAINING = 21_000
MAX_GAME_LENGTH = 60
BOARD_SIZE = 64
PADDING_TOKEN = 0

test_dataset = OthelloDataset(train=False, path="../dataset/test_dataset.pt")
intervention_dataset = Subset(test_dataset, range(NUM_OF_GAMES_FOR_PROBE_TRAINING, NUM_OF_GAMES_FOR_PROBE_TRAINING + NUM_OF_GAMES_USED_FOR_INTERVENTIONS))

# Intervention

In [43]:
def load_probe_checkpoint(probe, checkpoint):
    checkpoint = torch.load("./probes/" + checkpoint)
    probe.load_state_dict(checkpoint)

In [44]:
def get_position_based_on_token(token):
    flat_index = token - 1

    if flat_index >= 33:
        flat_index += 4
    elif flat_index >= 27:
        flat_index += 2

    row = flat_index // 8
    col = flat_index % 8
    position = GameBoard.index_to_position((row, col))

    return position

In [45]:
def print_current_turn_from_history(game_history):
    if not game_history:
        next_player = Piece.BLACK
    else:
        last_player = game_history[-1][0]
        next_player = Piece.WHITE if last_player == Piece.BLACK else Piece.BLACK

    board = GameBoard(game_history)

    if not board.get_legal_moves(next_player):
        next_player = Piece.WHITE if next_player == Piece.BLACK else Piece.BLACK

    return next_player

In [46]:
def get_intervention_vector(layer_num, board_position, target_pawn, model_name=model_name):
    """

    :param layer_num: GPT Layer
    :param board_position: For example "A4"
    :param target_pawn: "Mine" "Yours", "Empty"
    :param model_name: Name of the model used to compute the intervention vector and train the probes.
    :return: vector d
    """
    probe = LinearProbe(model.d_model, 64*3)
    load_probe_checkpoint(probe, f"linear_probe_layer_{layer_num}_{model_name}.pt")
    pawn_map = {"Mine": 0, "Yours": 1, "Empty": 2}

    # Get index
    row, col = GameBoard.position_to_index(board_position)
    flat_index = row * 8 + col

    # Get pawn
    pawn = pawn_map[target_pawn]

    target_idx = flat_index * 3 + pawn

    with torch.no_grad():
        vector = probe.linear.weight[target_idx, :].clone()

    return vector

In [48]:
def convert_tensor_to_game_history(tensor, padding_token=0):
    if isinstance(tensor, torch.Tensor):
        moves = tensor.flatten().tolist()
    else:
        moves = tensor

    game_history = []

    board = GameBoard()
    current_player = Piece.BLACK

    for move_val in moves:
        if move_val == padding_token:
            break

        position = get_position_based_on_token(move_val)

        legal_moves = board.get_legal_moves(current_player)

        if not legal_moves:
            current_player = Piece.WHITE if current_player == Piece.BLACK else Piece.BLACK

        game_history.append((current_player, position))

        board.add_piece(current_player, position)

        current_player = Piece.WHITE if current_player == Piece.BLACK else Piece.BLACK

    return game_history

In [49]:
def get_prediction(tensor, model, padding_token=0):
    model.eval()

    if tensor.dim() == 2 and tensor.shape[1] == 1:
        x = tensor.squeeze(1).unsqueeze(0)
    elif tensor.dim() == 1:
        x = tensor.unsqueeze(0)
    else:
        x = tensor

    device = next(model.parameters()).device
    x = x.to(device)

    non_padding_mask = (x != padding_token)
    valid_indices = non_padding_mask.nonzero(as_tuple=True)[1]
    last_idx = valid_indices[-1].item() if len(valid_indices) > 0 else 0
    x_trimmed = x[:, :last_idx + 1]


    with torch.no_grad():
        logits = model.forward_with_intervention(x_trimmed, None)

    last_step_logits = logits[0, -1, :]
    predicted_index = last_step_logits.argmax(dim=-1).item()

    return predicted_index

In [50]:
def get_prediction_with_intervention(tensor, model, intervention_vectors, padding_token=0):
    model.eval()

    if tensor.dim() == 2 and tensor.shape[1] == 1:
        x = tensor.squeeze(1).unsqueeze(0)
    elif tensor.dim() == 1:
        x = tensor.unsqueeze(0)
    else:
        x = tensor

    device = next(model.parameters()).device
    x = x.to(device)

    if intervention_vectors is not None:
        intervention_vectors = [
            v.to(device) if v is not None else None
            for v in intervention_vectors
        ]

    non_padding_mask = (x != padding_token)
    valid_indices = non_padding_mask.nonzero(as_tuple=True)[1]
    last_idx = valid_indices[-1].item() if len(valid_indices) > 0 else 0
    x_trimmed = x[:, :last_idx + 1]

    with torch.no_grad():
        logits = model.forward_with_intervention(x_trimmed, intervention_vectors)

    last_step_logits = logits[0, -1, :]
    predicted_index = last_step_logits.argmax(dim=-1).item()

    return predicted_index

In [51]:
x, y = intervention_dataset[2]
x[3:] = 0

game = GameBoard(convert_tensor_to_game_history(x))

print("Current prediction (without intervention):")
game.display([get_position_based_on_token(get_prediction(x, model))])

print(f"Possible moves for current player: {print_current_turn_from_history(game.game_history)}")
game.display(game.get_legal_moves(print_current_turn_from_history(game.game_history)))

b = "D6"
pawn = "Mine"
print(f"Predicted move after intervention. Cell: {b}, Pawn: {pawn}")
intervention_vectors = [2.3 * get_intervention_vector(i, b, pawn) for i in range(2, 5)]
game.display([get_position_based_on_token(get_prediction_with_intervention(x, model, intervention_vectors))])

Current prediction (without intervention):
   A  B  C  D  E  F  G  H
1  ·  ·  ·  ·  ·  ·  ·  ·
2  ·  ·  ·  ·  ·  ·  ·  ·
3  ·  ·  ●  ●  ·  ·  ·  ·
4  ·  ·  ·  ●  ●  ·  ·  ·
5  ·  ·  ·  ●  ●  ●  ·  ·
6  ·  ·  ·  ·  ·  ·  ·  ·
7  ·  ·  ·  ·  ·  ·  ·  ·
8  ·  ·  ·  ·  ·  ·  ·  ·
Possible moves for current player: ●
   A  B  C  D  E  F  G  H
1  ·  ·  ·  ·  ·  ·  ·  ·
2  ·  ·  ·  ·  ·  ·  ·  ·
3  ·  ·  ●  ●  ·  ·  ·  ·
4  ·  ·  ·  ●  ●  ·  ·  ·
5  ·  ·  ·  ●  ●  ●  ·  ·
6  ·  ·  ·  ·  ·  ·  ·  ·
7  ·  ·  ·  ·  ·  ·  ·  ·
8  ·  ·  ·  ·  ·  ·  ·  ·
Predicted move after intervention. Cell: D6, Pawn: Mine
   A  B  C  D  E  F  G  H
1  ·  ·  ·  ·  ·  ·  ·  ·
2  ·  ·  ·  ·  ·  ·  ·  ·
3  ·  ·  ●  ●  ·  ·  ·  ·
4  ·  ·  ·  ●  ●  ·  ·  ·
5  ·  ·  ·  ●  ●  ●  ·  ·
6  ·  ·  ·  ·  ·  ·  ·  ·
7  ·  ·  ·  ·  ·  ·  ·  ·
8  ·  ·  ·  ·  ·  ·  ·  ·


In [53]:
def get_current_turn_silent(game_history):
    if not game_history:
        next_player = Piece.BLACK
    else:
        last_player = game_history[-1][0]
        next_player = Piece.WHITE if last_player == Piece.BLACK else Piece.BLACK

    board = GameBoard(game_history)
    if not board.get_legal_moves(next_player):
        next_player = Piece.WHITE if next_player == Piece.BLACK else Piece.BLACK

    return next_player

def get_top_n_predictions_with_intervention(tensor, model, intervention_vectors, n, padding_token=0):
    if n == 0:
        return []

    model.eval()

    if tensor.dim() == 2 and tensor.shape[1] == 1:
        x = tensor.squeeze(1).unsqueeze(0)
    elif tensor.dim() == 1:
        x = tensor.unsqueeze(0)
    else:
        x = tensor

    device = next(model.parameters()).device
    x = x.to(device)

    if intervention_vectors is not None:
        intervention_vectors = [
            v.to(device) if v is not None else None
            for v in intervention_vectors
        ]

    non_padding_mask = (x != padding_token)
    valid_indices = non_padding_mask.nonzero(as_tuple=True)[1]
    last_idx = valid_indices[-1].item() if len(valid_indices) > 0 else 0
    x_trimmed = x[:, :last_idx + 1]

    with torch.no_grad():
        logits = model.forward_with_intervention(x_trimmed, intervention_vectors)

    last_step_logits = logits[0, -1, :].clone()

    last_step_logits[padding_token] = float('-inf')

    top_indices = last_step_logits.topk(n).indices.tolist()
    return [get_position_based_on_token(idx) for idx in top_indices]

def evaluate_intervention_deterministic(dataset, model, n_games=50, layers_to_patch=None, alpha=1.0):
    total_errors = 0
    valid_scenarios = 0

    for game_idx in tqdm(range(n_games), desc=f"Eval (Alpha={alpha})"):
        x, _ = dataset[game_idx]
        non_padding = (x != 0).nonzero(as_tuple=True)[0]

        for t in range(10, len(non_padding)):
            x_partial = x.clone()
            x_partial[t+1:] = 0

            game_history = convert_tensor_to_game_history(x_partial)
            current_player = get_current_turn_silent(game_history)
            board = GameBoard(game_history)
            board_state = board.get_board()

            filled_tiles = []
            for r in range(8):
                for c in range(8):
                    if board_state[r, c] != Piece.EMPTY:
                        filled_tiles.append((GameBoard.index_to_position((r, c)), board_state[r, c]))

            if not filled_tiles:
                continue

            target_pos, original_piece = filled_tiles[-1]

            if original_piece == current_player:
                target_pawn_dir, new_piece = "Yours", (Piece.WHITE if current_player == Piece.BLACK else Piece.BLACK)
            else:
                target_pawn_dir, new_piece = "Mine", current_player

            board_prime = GameBoard(game_history)
            board_prime.add_piece_without_flip(new_piece, target_pos)

            legal_moves_prime = board_prime.get_legal_moves(current_player)
            N_moves = len(legal_moves_prime)

            if N_moves == 0:
                continue

            intervention_vectors = [None] * (model.n_layers + 1)
            for layer in layers_to_patch:
                v = get_intervention_vector(layer, target_pos, target_pawn_dir)
                if v is not None:
                    intervention_vectors[layer] = alpha * (v / (v.norm() + 1e-8))

            predicted_moves = get_top_n_predictions_with_intervention(x_partial, model, intervention_vectors, N_moves)

            set_true = set(legal_moves_prime)
            set_pred = set(predicted_moves)

            false_positives = len(set_pred - set_true)
            false_negatives = len(set_true - set_pred)
            error_count = false_positives + false_negatives

            total_errors += error_count
            valid_scenarios += 1

    avg_error_rate = total_errors / valid_scenarios if valid_scenarios > 0 else 0

    return avg_error_rate

In [ ]:
from tqdm import tqdm

def get_contiguous_blocks(num_layers):
    blocks = []
    for start in range(num_layers):
        for end in range(start, num_layers):
            blocks.append(list(range(start, end + 1)))
    return blocks

all_layer_blocks = get_contiguous_blocks(4)
alphas = [2.7]

results = []
best_f1 = -1.0
best_params = {}

for layers in all_layer_blocks:
    for a in alphas:
        precision, recall = evaluate_intervention_deterministic(
            dataset=intervention_dataset,
            model=model,
            n_games=40,
            layers_to_patch=layers,
            alpha=a
        )

        if (precision + recall) > 0:
            f1 = 2 * (precision * recall) / (precision + recall)
        else:
            f1 = 0

        results.append({
            "layers": layers,
            "alpha": round(a, 2),
            "precision": round(precision, 4),
            "recall": round(recall, 4),
            "f1": round(f1, 4)
        })

        if f1 > best_f1:
            best_f1 = f1
            best_params = {
                "layers": layers,
                "alpha": round(a, 2),
                "precision": precision,
                "recall": recall
            }
            print(f"🔥 NEW BEST! F1: {f1:.4f} | P: {precision:.4f} | R: {recall:.4f} | Layers: {layers} | Alpha: {a:.1f}")

print(f"📍 Layers: {best_params['layers']}")
print(f"📍 Alpha: {best_params['alpha']}")
print(f"📈 Precision: {best_params['precision']:.44f}")
print(f"📉 Recall: {best_params['recall']:.4f}")

In [55]:
error = evaluate_intervention_deterministic(
            dataset=intervention_dataset,
            model=model,
            n_games=100,
            layers_to_patch=[1,2,3],
            alpha=2.7
        )
print(error)

Eval (Alpha=2.7): 100%|██████████| 100/100 [01:17<00:00,  1.28it/s]

0.5519493774239641


In [57]:
error = evaluate_intervention_deterministic(
            dataset=intervention_dataset,
            model=model,
            n_games=100,
            layers_to_patch=[],
            alpha=0.0
        )
print(error)

Eval (Alpha=0.0): 100%|██████████| 100/100 [01:05<00:00,  1.52it/s]

1.1390079608083283
